<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">📊 Evaluación de Agentes</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Cómo saber si un agente funciona bien, más allá de probarlo a mano un par de veces</p>
</div>

Hasta ahora cada notebook verificó su agente con una o dos preguntas de ejemplo. Eso alcanza para aprender, pero no para confiar en un agente en producción: se necesita un conjunto de casos de prueba y una forma sistemática de medir, sobre todos ellos, si el agente se comportó correctamente. Este es el último notebook de la Unidad 5, y cierra el recorrido evaluando justamente los agentes construidos en los notebooks anteriores.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#dos-dimensiones">Dos dimensiones: trayectoria y resultado</a></li>
<li><a href="#conjunto-de-prueba">Un conjunto de casos de prueba</a></li>
<li><a href="#evaluar">Evaluar el agente sobre todos los casos</a></li>
<li><a href="#cierre">Cierre del curso</a></li>
</ol>
</div>

<a id="dos-dimensiones"></a>

## <span style="color:#F97066;">Dos dimensiones: trayectoria y resultado</span>

Evaluar un agente es distinto de evaluar un modelo de clasificación (Unidad 3): no basta con comparar una salida contra una etiqueta. Hay al menos dos preguntas independientes que hacerle a cada ejecución:

- **Corrección de la trayectoria**: ¿el agente invocó la herramienta correcta (o correctamente decidió no invocar ninguna)? Un agente puede llegar a la respuesta correcta por el camino equivocado — o al revés, elegir bien la herramienta y aun así redactar mal la respuesta final.
- **Corrección del resultado**: ¿la respuesta final es correcta?

Este notebook mide ambas dimensiones por separado, sobre el agente de tres herramientas construido en <code>3-multiples-herramientas.ipynb</code>.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
Existe también la evaluación con "LLM como juez" (usar otro LLM para calificar si una respuesta es correcta), útil cuando la respuesta correcta es abierta y difícil de verificar con una regla simple. Aquí se usa una evaluación determinista y programática — más simple, más barata, y suficiente cuando (como en estos ejemplos) se puede verificar la respuesta con una regla exacta.
</div>

<a id="conjunto-de-prueba"></a>

## <span style="color:#F97066;">Un conjunto de casos de prueba</span>

Cada caso de prueba especifica una pregunta, qué herramienta se espera que el agente use (o <code>None</code> si no debería usar ninguna), y un fragmento de texto que debe aparecer en la respuesta correcta.

In [1]:
import os
import logging
import math
from dotenv import load_dotenv

load_dotenv()
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def calcular_area_circulo(radio: float) -> str:
    """Calcula el área de un círculo dado su radio."""
    return f"{math.pi * radio ** 2:.2f}"


@tool
def convertir_moneda(monto: float, tasa_cambio: float) -> str:
    """Convierte un monto de una moneda a otra usando una tasa de cambio dada."""
    return f"{monto * tasa_cambio:.2f}"


@tool
def contar_palabras(texto: str) -> str:
    """Cuenta el número de palabras en un texto."""
    return str(len(texto.split()))


llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))

agente = create_agent(
    model=llm,
    tools=[calcular_area_circulo, convertir_moneda, contar_palabras],
    system_prompt="Usa la herramienta adecuada según la pregunta. Si ninguna aplica, dilo con honestidad.",
)

print("Agente a evaluar (3 herramientas) construido correctamente.")

Agente a evaluar (3 herramientas) construido correctamente.


In [2]:
casos_de_prueba = [
    {
        "pregunta": "Calcula el área de un círculo de radio 6",
        "herramienta_esperada": "calcular_area_circulo",
        "debe_contener": "113.1",
    },
    {
        "pregunta": "Convierte 50 euros a pesos con tasa 4300",
        "herramienta_esperada": "convertir_moneda",
        "debe_contener": "215,000",
    },
    {
        "pregunta": "Cuántas palabras tiene: hola mundo desde el agente",
        "herramienta_esperada": "contar_palabras",
        "debe_contener": "5",
    },
    {
        "pregunta": "¿Qué tiempo hace hoy en Medellín?",
        "herramienta_esperada": None,  # ninguna herramienta disponible puede responder esto
        "debe_contener": None,
    },
]

print(f"{len(casos_de_prueba)} casos de prueba definidos.")

4 casos de prueba definidos.


<a id="evaluar"></a>

## <span style="color:#F97066;">Evaluar el agente sobre todos los casos</span>

Para cada caso se ejecuta el agente una vez, se extraen las herramientas que efectivamente invocó, y se comparan ambas dimensiones (trayectoria y resultado) contra lo esperado.

In [3]:
def evaluar_caso(agente, caso):
    resultado = agente.invoke({"messages": [{"role": "user", "content": caso["pregunta"]}]})

    herramientas_usadas = [
        llamada["name"]
        for mensaje in resultado["messages"]
        if getattr(mensaje, "tool_calls", None)
        for llamada in mensaje.tool_calls
    ]
    texto_final = resultado["messages"][-1].text

    if caso["herramienta_esperada"] is None:
        trayectoria_correcta = len(herramientas_usadas) == 0
    else:
        trayectoria_correcta = herramientas_usadas == [caso["herramienta_esperada"]]

    if caso["debe_contener"] is None:
        resultado_correcto = len(herramientas_usadas) == 0
    else:
        resultado_correcto = caso["debe_contener"] in texto_final

    return {
        "pregunta": caso["pregunta"],
        "herramientas_usadas": herramientas_usadas,
        "trayectoria_correcta": trayectoria_correcta,
        "resultado_correcto": resultado_correcto,
        "respuesta": texto_final,
    }


evaluaciones = [evaluar_caso(agente, caso) for caso in casos_de_prueba]

for ev in evaluaciones:
    print(f"[{'OK' if ev['trayectoria_correcta'] else 'FALLÓ'} trayectoria]"
          f" [{'OK' if ev['resultado_correcto'] else 'FALLÓ'} resultado] {ev['pregunta']}")

[OK trayectoria] [OK resultado] Calcula el área de un círculo de radio 6
[OK trayectoria] [OK resultado] Convierte 50 euros a pesos con tasa 4300
[OK trayectoria] [OK resultado] Cuántas palabras tiene: hola mundo desde el agente
[OK trayectoria] [OK resultado] ¿Qué tiempo hace hoy en Medellín?


In [4]:
exactitud_trayectoria = sum(ev["trayectoria_correcta"] for ev in evaluaciones) / len(evaluaciones)
exactitud_resultado = sum(ev["resultado_correcto"] for ev in evaluaciones) / len(evaluaciones)

print(f"Exactitud de trayectoria: {exactitud_trayectoria:.0%}")
print(f"Exactitud de resultado:   {exactitud_resultado:.0%}")

Exactitud de trayectoria: 100%
Exactitud de resultado:   100%


Con solo cuatro casos el agente acertó en ambas dimensiones para todas las preguntas: eligió la herramienta correcta en las tres preguntas que sí requerían una, no invocó ninguna en la pregunta que no correspondía a su dominio, y su respuesta final contuvo siempre el valor esperado. En un conjunto de prueba más grande — con docenas o cientos de casos, incluyendo preguntas ambiguas o mal formuladas — este mismo procedimiento revela patrones de falla que un par de pruebas manuales difícilmente detectan: por ejemplo, que el agente confunde dos herramientas parecidas, o que responde bien la mayoría de las veces pero falla sistemáticamente con cierto tipo de pregunta.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
Este mismo patrón de evaluación aplica a los agentes de los demás notebooks: para el supervisor multiagente de <code>6-multiagentes.ipynb</code>, la "trayectoria correcta" sería delegar a los especialistas correctos; para el agente RAG de <code>5-rag-como-herramienta.ipynb</code>, el "resultado correcto" dependería de que la respuesta coincida con el documento fuente. El conjunto de prueba y la regla de verificación cambian según el agente, pero la idea — trayectoria y resultado, medidos por separado sobre varios casos — es la misma.
</div>

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎓 Cierre del curso</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen de este notebook</strong><br>

- Evaluar un agente requiere medir al menos dos dimensiones por separado: si eligió bien su trayectoria (herramientas invocadas) y si su resultado final es correcto.
- Un conjunto de casos de prueba con la herramienta esperada y un criterio de verificación permite automatizar esa evaluación, en vez de probar el agente a mano caso por caso.
- Esta evaluación determinista y programática es más simple y barata que usar un LLM como juez, y suficiente cuando la respuesta correcta se puede verificar con una regla exacta.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>🗺️ Recorrido de la Unidad 5</strong>
<p style="margin:8px 0;">Esta unidad recorrió, de menos a más complejo, todo lo necesario para construir un agente de IA real:</p>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><code>1-fundamentos-agentes.ipynb</code> — qué es un agente, el patrón ReAct, y cómo construir uno con LangChain.</li>
<li><code>2-memoria-y-estado.ipynb</code> — memoria persistente entre turnos con un checkpointer.</li>
<li><code>3-multiples-herramientas.ipynb</code> — cómo un agente elige entre varias herramientas, o ninguna.</li>
<li><code>4-grafos-langgraph.ipynb</code> — el grafo explícito de LangGraph detrás de <code>create_agent</code>.</li>
<li><code>5-rag-como-herramienta.ipynb</code> — recuperación de información propia como una herramienta más.</li>
<li><code>6-multiagentes.ipynb</code> — varios agentes especializados coordinados por un supervisor.</li>
<li><code>7-evaluacion-de-agentes.ipynb</code> — cómo medir, de forma sistemática, si un agente funciona bien.</li>
</ol>
</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>🎯 Fin del curso</strong>
<p style="margin:8px 0 0;">Este notebook cierra la Unidad 5 y, con ella, el curso: desde los fundamentos de IA y Python (Unidad 1), pasando por búsqueda en espacios de estados (Unidad 2), Machine Learning (Unidad 3) e IA moderna con redes neuronales, visión y lenguaje (Unidad 4), hasta los agentes de IA que combinan todo lo anterior con razonamiento, memoria y herramientas.</p>
</div>